# Interpretabilidade do Modelo Selecionado

Após a comparação entre os algoritmos avaliados, o LightGBM foi selecionado como o modelo final para a classificação da qualidade da água no AquaSense. Entretanto, embora as métricas de desempenho permitam identificar qual modelo apresenta melhores resultados, elas não explicam diretamente como as previsões são realizadas.

Em aplicações relacionadas ao monitoramento ambiental, compreender os fatores que influenciam as decisões do modelo é tão importante quanto alcançar bons resultados de classificação. Essa análise permite verificar se o algoritmo está utilizando padrões coerentes com o conhecimento do domínio e aumenta a transparência do processo de tomada de decisão.

Para essa finalidade, foi utilizada a técnica **SHAP (SHapley Additive exPlanations)**, uma abordagem baseada na Teoria dos Jogos que permite quantificar a contribuição individual de cada variável para as previsões realizadas pelo modelo.

De forma simplificada, os valores SHAP indicam quanto cada atributo contribui para aumentar ou diminuir a probabilidade de determinada classificação. Dessa forma, é possível identificar quais variáveis exercem maior influência sobre as decisões do algoritmo e compreender melhor os padrões aprendidos durante o treinamento.

Nesta seção serão apresentados os resultados da análise de interpretabilidade do LightGBM, incluindo a importância global das variáveis e o impacto de cada atributo nas previsões realizadas pelo modelo.

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import shap
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score
)


from lightgbm import LGBMClassifier

import warnings

warnings.filterwarnings("ignore")

SEED = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_PATH = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_final.parquet"
else:
    DATA_PATH = "../../dataset/processed/amostra_rotulada_final.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape: {df.shape}")
print(f"\nDistribuição do rótulo:")
print(df["conama_status"].value_counts())

Mounted at /content/drive
Dataset carregado com sucesso.
Shape: (59896, 27)

Distribuição do rótulo:
conama_status
Atenção         29946
Adequada        20585
Não adequada     9365
Name: count, dtype: int64


In [3]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year,prob_adequada,prob_nao_adequada,subgrupo_qualidade,conama_status_revisado
0,Canada,FISW_32,Lake,2003-12-02,0.0438,2.1333,9.8240,0.0020,7.7900,12.0000,0.2300,2.3552,100.0000,Excellent,1,1,1,1,2.0000,1,5,Adequada,2003,0.9717,0.0283,Melhores adequadas,Adequada
1,Canada,IEEA_10_32,Lake,2001-06-08,0.0159,0.5500,9.8240,0.0040,7.7900,16.8000,0.0070,0.6242,100.0000,Excellent,1,1,1,1,2.0000,1,5,Adequada,2001,0.9714,0.0286,Melhores adequadas,Adequada
2,Canada,CHRW-1876,River,2000-01-12,0.0644,10.8750,11.2500,0.0359,8.2833,12.7615,0.4000,5.7042,92.2731,Good,1,1,0,1,1.0000,1,4,Atenção,2000,0.9078,0.0922,Melhores não adequadas,Atenção
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.0717,1.2444,5.8500,0.2042,7.1000,18.3250,0.4000,20.1683,79.7143,Fair,1,1,1,0,3.7000,1,4,Atenção,2004,0.8036,0.1964,Melhores não adequadas,Atenção
4,Canada,CZPLA_391,River,2003-01-12,0.0397,1.8333,11.0500,0.0610,7.7500,8.6667,0.4000,10.4768,93.1167,Good,1,1,1,0,2.0000,1,4,Atenção,2003,0.9080,0.0920,Melhores não adequadas,Atenção


In [4]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [5]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [6]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [7]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [8]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

In [9]:
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.9204858502379164
Train Precision:
0.9262303198976285
Train Recall:
0.9204858502379164
Train F1:
0.9210923681889642

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.94      0.98      0.96     16468
     Atenção       0.96      0.87      0.92     23956
Não adequada       0.78      0.93      0.85      7492

    accuracy                           0.92     47916
   macro avg       0.89      0.93      0.91     47916
weighted avg       0.93      0.92      0.92     47916

Train Confusion Matrix:
[[16192   276     0]
 [ 1086 20941  1929]
 [    0   519  6973]]


In [10]:
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9042570951585976

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.93      0.97      0.95      4117
     Atenção       0.94      0.86      0.90      5990
Não adequada       0.75      0.89      0.82      1873

    accuracy                           0.90     11980
   macro avg       0.88      0.91      0.89     11980
weighted avg       0.91      0.90      0.91     11980


Confusion Matrix:
[[4010  107    0]
 [ 295 5149  546]
 [   0  199 1674]]


In [11]:
import joblib

joblib.dump(
    model,
    "aquasense_modelo_final.joblib"
)

['aquasense_modelo_final.joblib']

In [12]:
import joblib

joblib.dump(
    model,
    "/content/drive/MyDrive/EDA_AquaSense/aquasense_modelo_final.joblib"
)

['/content/drive/MyDrive/EDA_AquaSense/aquasense_modelo_final.joblib']

In [13]:
features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]

joblib.dump(features, "/content/drive/MyDrive/EDA_AquaSense/features_modelo.joblib")

['/content/drive/MyDrive/EDA_AquaSense/features_modelo.joblib']

In [14]:
import plotly.graph_objects as go

# =====================================
# RESULTADOS CONFIGURAÇÃO FINAL
# =====================================

metricas = {
    "Precisão\nAdequada": {
        "Treino": 94,
        "Teste": 93
    },
    "Precisão\nAtenção": {
        "Treino": 96,
        "Teste": 94
    },
    "Precisão\nNão Adequada": {
        "Treino": 78,
        "Teste": 75
    },

    "Recall\nAdequada": {
        "Treino": 98,
        "Teste": 97
    },
    "Recall\nAtenção": {
        "Treino": 87,
        "Teste": 86
    },
    "Recall\nNão Adequada": {
        "Treino": 93,
        "Teste": 89
    },

    "F1\nAdequada": {
        "Treino": 96,
        "Teste": 95
    },
    "F1\nAtenção": {
        "Treino": 92,
        "Teste": 90
    },
    "F1\nNão Adequada": {
        "Treino": 85,
        "Teste": 82
    }
}

categorias = list(metricas.keys())
treino = [metricas[m]["Treino"] for m in categorias]
teste = [metricas[m]["Teste"] for m in categorias]

# =====================================
# CORES AQUASENSE
# =====================================

COR_TREINO = "#3FF3D7"
COR_TESTE = "#004D48"

# =====================================
# GRÁFICO
# =====================================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=categorias,
        y=treino,
        name="Treino",
        marker_color=COR_TREINO,
        text=[f"{v:.0f}%" for v in treino],
        textposition="outside"
    )
)

fig.add_trace(
    go.Bar(
        x=categorias,
        y=teste,
        name="Teste",
        marker_color=COR_TESTE,
        text=[f"{v:.0f}%" for v in teste],
        textposition="outside"
    )
)

fig.update_layout(
    title={
        "text": "Configuração Final - Desempenho por Classe",
        "x": 0.5,
        "font": dict(
            size=24,
            color="#004D48"
        )
    },
    barmode="group",
    height=650,
    width=1300,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Arial",
        size=13,
        color="#004D48"
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.10
    ),
    margin=dict(
        l=50,
        r=50,
        t=100,
        b=120
    )
)

fig.update_yaxes(
    title="Percentual (%)",
    range=[0, 105],
    showgrid=True,
    gridcolor="#DDEFEA"
)

fig.show()

# Cálculo dos Valores SHAP

Após o treinamento do modelo LightGBM utilizando a estrutura de classes reconstruídas proposta neste trabalho, foram calculados os valores SHAP (*SHapley Additive exPlanations*) com o objetivo de interpretar a contribuição de cada variável para as previsões realizadas pelo algoritmo. Essa etapa permite compreender quais atributos exerceram maior influência na separação das classes **Adequada**, **Atenção** e **Não Adequada**, fornecendo uma visão mais transparente do processo decisório do modelo.

Como o problema tratado corresponde a uma classificação multiclasse, os valores SHAP são calculados individualmente para cada classe prevista. Dessa forma, antes da construção das visualizações e da realização das análises interpretativas, torna-se necessário verificar a estrutura dos valores retornados pela biblioteca SHAP, uma vez que diferentes versões da ferramenta podem apresentar formatos distintos para o armazenamento das contribuições associadas a cada classe.


In [15]:
model.named_steps

{'preprocessor': ColumnTransformer(remainder='passthrough',
                   transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                  ['Country', 'Waterbody Type'])]),
 'classifier': LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1)}

In [16]:
lgbm_model = model.named_steps["classifier"]

print(type(lgbm_model))

<class 'lightgbm.sklearn.LGBMClassifier'>


In [17]:
preprocessor = model.named_steps["preprocessor"]

X_test_transformed = preprocessor.transform(X_test)

print(X_test_transformed.shape)

(11980, 19)


In [18]:
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed_dense = X_test_transformed.toarray()
else:
    X_test_transformed_dense = X_test_transformed

X_test_transformed_final = X_test_transformed_dense.astype(float)

explainer = shap.Explainer(
    lgbm_model,
    X_test_transformed_final
)

shap_values = explainer(
    X_test_transformed_final,
    check_additivity=False
)

print(type(shap_values))
print(shap_values.values.shape)

100%|===================| 35902/35940 [07:14<00:00]       

<class 'shap._explanation.Explanation'>
(11980, 19, 3)


In [19]:
print(shap_values.values.shape)

(11980, 19, 3)


In [20]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))

for i, feature in enumerate(feature_names):
    print(i, feature)

19
0 cat__Country_Canada
1 cat__Country_China
2 cat__Country_England
3 cat__Country_Ireland
4 cat__Country_USA
5 cat__Waterbody Type_Bay
6 cat__Waterbody Type_Canal
7 cat__Waterbody Type_Drainage
8 cat__Waterbody Type_Effluent
9 cat__Waterbody Type_Estuarine
10 cat__Waterbody Type_Lake
11 cat__Waterbody Type_Marine
12 cat__Waterbody Type_River
13 cat__Waterbody Type_Sea Water
14 cat__Waterbody Type_Sewage
15 cat__Waterbody Type_Transitional
16 remainder__Temperature (cel)
17 remainder__Orthophosphate (mg/l)
18 remainder__Nitrogen (mg/l)


In [21]:
feature_names = preprocessor.get_feature_names_out()

importance = np.abs(shap_values.values).mean(axis=(0, 2))

shap_importance = pd.DataFrame({
    "Variável": feature_names,
    "Importância SHAP": importance
}).sort_values(
    by="Importância SHAP",
    ascending=False
)

shap_importance

,Variável,Importância SHAP
17,remainder__Orthophosphate (mg/l),1.0149
18,remainder__Nitrogen (mg/l),0.9016
8,cat__Waterbody Type_Effluent,0.6039
16,remainder__Temperature (cel),0.2909
12,cat__Waterbody Type_River,0.1113
14,cat__Waterbody Type_Sewage,0.0942
9,cat__Waterbody Type_Estuarine,0.0625
4,cat__Country_USA,0.0591
3,cat__Country_Ireland,0.0400
6,cat__Waterbody Type_Canal,0.0302


In [22]:
import plotly.graph_objects as go
import pandas as pd

# =========================
# IMPORTÂNCIA SHAP
# =========================

dados_shap = {
    "Variável": [
        "Orthophosphate (mg/l)",
        "Nitrogen (mg/l)",
        "Waterbody Type - Effluent",
        "Temperature (cel)",
        "Waterbody Type - River",
        "Waterbody Type - Sewage",
        "Waterbody Type - Estuarine",
        "Country - USA",
        "Country - Ireland",
        "Waterbody Type - Canal",
        "Country - England",
        "Waterbody Type - Drainage",
        "Waterbody Type - Lake",
        "Waterbody Type - Sea Water",
        "Country - China",
        "Country - Canada",
        "Waterbody Type - Bay",
        "Waterbody Type - Marine",
        "Waterbody Type - Transitional"
    ],
    "Importância SHAP": [
        1.0149,
        0.9016,
        0.6039,
        0.2909,
        0.1113,
        0.0942,
        0.0625,
        0.0591,
        0.0400,
        0.0302,
        0.0100,
        0.0083,
        0.0051,
        0.0035,
        0.0023,
        0.0001,
        0.0000,
        0.0000,
        0.0000
    ]
}

df_shap = pd.DataFrame(dados_shap)

# inverter para maior aparecer no topo no gráfico horizontal
df_plot = df_shap.sort_values("Importância SHAP", ascending=True)

# =========================
# GRÁFICO
# =========================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_plot["Importância SHAP"],
        y=df_plot["Variável"],
        orientation="h",
        marker_color="#004D48",
        text=df_plot["Importância SHAP"].apply(lambda x: f"{x:.4f}"),
        textposition="outside"
    )
)

fig.update_layout(
    title={
        "text": "Importância Global das Variáveis pelo SHAP",
        "x": 0.5,
        "font": dict(size=24, color="#004D48")
    },
    height=750,
    width=1100,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Arial",
        size=13,
        color="#004D48"
    ),
    margin=dict(l=260, r=80, t=90, b=60)
)

fig.update_xaxes(
    title="Importância SHAP média",
    showgrid=True,
    gridcolor="#DDEFEA"
)

fig.update_yaxes(
    title="",
    tickfont=dict(size=12)
)

fig.show()

## Interpretação da Importância Global das Variáveis

A análise SHAP foi utilizada para compreender quais variáveis exerceram maior influência nas decisões do modelo LightGBM treinado sobre a estrutura de classes reconstruídas proposta neste trabalho. Diferentemente das abordagens anteriores, em que as classes eram definidas diretamente a partir dos níveis de qualidade da água, a configuração final adotada foi construída por meio da reconstrução probabilística dos rótulos. Nesse processo, as probabilidades produzidas pelo modelo binário foram utilizadas para definir subclasses internas e, posteriormente, formar as três classes finais: **Adequada**, **Atenção** e **Não Adequada**. Assim, os valores SHAP devem ser interpretados como indicadores das variáveis que mais contribuíram para a separação dessa nova estrutura de classes reconstruídas.

Os resultados indicaram que **Orthophosphate (mg/l)**, **Nitrogen (mg/l)**, **Waterbody Type_Effluent**, **Temperature (cel)** e **Waterbody Type_River** foram os atributos com maior influência nas previsões realizadas pelo modelo. Essas variáveis apresentaram os maiores valores médios absolutos de SHAP, indicando que exerceram maior impacto sobre as decisões do algoritmo ao distinguir os diferentes grupos definidos pela estratégia de reconstrução dos rótulos.

É importante destacar que os valores SHAP não representam relações de causa e efeito. Essa técnica apenas quantifica a contribuição de cada variável para as previsões do modelo, permitindo identificar quais atributos foram mais úteis para a separação das classes aprendidas. Dessa forma, a interpretação dos resultados deve ser realizada em conjunto com as análises exploratórias previamente desenvolvidas e com o conhecimento consolidado na literatura sobre monitoramento ambiental.

As duas variáveis mais importantes identificadas pelo modelo foram **Orthophosphate (mg/l)** e **Nitrogen (mg/l)**, com valores médios absolutos de SHAP de 1,0149 e 0,9016, respectivamente. Esse resultado é consistente com os achados obtidos durante a etapa de análise exploratória dos dados, na qual esses parâmetros já haviam se destacado entre os atributos mais associados às variações do índice CCME. Além disso, foi observada anteriormente uma correlação positiva entre Orthophosphate e Nitrogen, indicando que ambos frequentemente variam em conjunto e carregam informações complementares sobre as condições ambientais dos corpos hídricos analisados.

A relevância dessas variáveis torna-se ainda mais significativa quando considerada a estratégia de reconstrução dos rótulos adotada neste trabalho. Como a classe intermediária **Atenção** foi formada a partir das regiões de maior incerteza identificadas pelo modelo binário, enquanto as classes **Adequada** e **Não Adequada** representam situações de maior confiança classificatória, os resultados sugerem que Orthophosphate e Nitrogen foram determinantes para a definição dessas fronteiras probabilísticas. Em outras palavras, esses atributos não contribuíram apenas para distinguir condições extremas, mas também para caracterizar as situações intermediárias que compõem a nova estrutura de classes.

A literatura especializada reforça essa interpretação. Nutrientes como fósforo e nitrogênio são amplamente utilizados em estudos de monitoramento ambiental devido à sua associação com processos de enriquecimento nutricional, eutrofização e degradação da qualidade da água. Dessa forma, embora o modelo não possua conhecimento explícito sobre os processos ambientais envolvidos, a convergência entre os padrões identificados pelo algoritmo e o conhecimento consolidado na literatura fortalece a plausibilidade das relações aprendidas.

A variável **Waterbody Type_Effluent** apresentou a terceira maior importância global do modelo, com valor SHAP médio de 0,6039. Esse resultado indica que a identificação de corpos hídricos classificados como efluentes forneceu informações relevantes para a separação das classes reconstruídas. Durante as análises exploratórias realizadas anteriormente, observou-se que categorias como *Effluent*, *Sewage* e *Drainage* frequentemente apresentavam valores mais elevados para parâmetros associados à degradação da qualidade da água. Dessa forma, é plausível que o algoritmo tenha utilizado essa característica como um importante elemento discriminativo para identificar padrões relacionados às diferentes condições ambientais presentes na base de dados.

Entretanto, é importante ressaltar que a importância observada para essa variável não implica que o tipo de corpo hídrico seja a causa direta das previsões realizadas. A influência capturada pelo SHAP pode refletir relações indiretas existentes entre essa categoria e outros atributos utilizados pelo modelo, além de características específicas da distribuição das amostras presentes no conjunto de treinamento.

A variável **Temperature (cel)** apresentou a quarta maior importância global, com valor SHAP médio de 0,2909. A temperatura influencia diversos processos físicos, químicos e biológicos em ambientes aquáticos, afetando o comportamento de diferentes parâmetros relacionados à qualidade da água. Além disso, as análises exploratórias mostraram que essa variável apresentava padrões distintos entre diferentes contextos ambientais e tipos de corpos hídricos, o que pode ter contribuído para sua relevância dentro da estrutura decisória aprendida pelo modelo.

Entre as variáveis categóricas, **Waterbody Type_River** também apresentou destaque, com valor SHAP médio de 0,1113. Uma possível interpretação é que rios possuam características ambientais particulares relacionadas ao transporte de nutrientes, ao recebimento de cargas poluidoras e à dinâmica hidrológica, tornando essa categoria útil para a diferenciação dos padrões presentes na base. Além disso, trata-se de uma das categorias mais representativas do conjunto de dados, o que pode ter favorecido o aprendizado de padrões específicos associados a esse tipo de corpo hídrico.

Outras variáveis, como **Waterbody Type_Sewage**, **Waterbody Type_Estuarine**, **Country_USA** e **Country_Ireland**, também apresentaram contribuições mensuráveis para as previsões, embora com impacto inferior ao observado para os atributos mais relevantes. Por outro lado, diversas categorias relacionadas a países e a determinados tipos de corpos hídricos apresentaram importância reduzida ou praticamente nula. Isso sugere que, dentro da estrutura aprendida pelo modelo, essas características contribuíram menos para a separação das classes quando comparadas aos parâmetros físico-químicos e às categorias ambientais mais influentes.

De maneira geral, os resultados do SHAP demonstram que as decisões do LightGBM foram fortemente influenciadas por variáveis relacionadas à concentração de nutrientes, às características dos corpos hídricos e às condições ambientais observadas. Mais importante ainda, observa-se uma convergência entre os padrões identificados pelo modelo, os resultados das análises exploratórias e o conhecimento consolidado na literatura. Embora essa convergência não permita estabelecer relações causais, ela fornece evidências de que o algoritmo aprendeu padrões coerentes com a estrutura dos dados e capazes de sustentar a separação das classes reconstruídas propostas neste trabalho.


# Importância das Variáveis por Classe

A análise de importância global apresentada anteriormente permite identificar quais variáveis exercem maior influência sobre o comportamento geral do modelo. Entretanto, essa visão agregada não evidencia como cada atributo contribui para a identificação das classes individuais aprendidas pelo algoritmo.

Considerando que a configuração final adotada neste trabalho é composta pelas classes **Adequada**, **Atenção** e **Não Adequada**, definidas a partir do processo de reconstrução probabilística dos rótulos, torna-se relevante investigar como as variáveis influenciam especificamente cada uma dessas categorias. Essa análise permite compreender quais atributos foram mais importantes para caracterizar os diferentes padrões presentes na estrutura de classes reconstruídas e como o modelo distinguiu situações de maior adequação, maior inadequação e a região intermediária de transição representada pela classe Atenção.

Dessa forma, nesta etapa são analisadas as contribuições médias das variáveis para cada classe individualmente por meio dos valores SHAP. Essa abordagem complementa a análise de importância global ao fornecer uma visão mais detalhada do comportamento do modelo, permitindo identificar quais características tiveram maior participação na separação de cada classe específica.


In [23]:
feature_names = preprocessor.get_feature_names_out()

classes = [
    "Adequada",
    "Atenção",
    "Não adequada"
]

heatmap_data = pd.DataFrame(
    index=feature_names
)

for i, classe in enumerate(classes):
    heatmap_data[classe] = np.abs(
        shap_values.values[:, :, i]
    ).mean(axis=0)

heatmap_data

,Adequada,Atenção,Não adequada
cat__Country_Canada,0.0000,0.0003,0.0000
cat__Country_China,0.0014,0.0056,0.0000
cat__Country_England,0.0184,0.0117,0.0000
cat__Country_Ireland,0.0339,0.0820,0.0041
cat__Country_USA,0.0441,0.1333,0.0000
cat__Waterbody Type_Bay,0.0000,0.0000,0.0000
cat__Waterbody Type_Canal,0.0362,0.0539,0.0006
cat__Waterbody Type_Drainage,0.0145,0.0038,0.0065
cat__Waterbody Type_Effluent,1.4248,0.2835,0.1034
cat__Waterbody Type_Estuarine,0.0689,0.0625,0.0561


## Interpretação da Importância das Variáveis por Classe

A análise da importância das variáveis por classe revelou que os atributos não exercem a mesma influência sobre todas as categorias da estrutura reconstruída. Embora **Orthophosphate (mg/l)** e **Nitrogen (mg/l)** tenham se destacado na análise global, observa-se que suas contribuições variam entre as classes **Adequada**, **Atenção** e **Não Adequada**, indicando que o modelo utiliza essas informações de maneiras distintas durante o processo de classificação.

A classe **Adequada** apresentou forte dependência de variáveis associadas aos nutrientes e ao contexto ambiental dos corpos hídricos. Os maiores valores de importância foram observados para **Waterbody Type_Effluent** (1,4248), **Orthophosphate (mg/l)** (1,3282) e **Nitrogen (mg/l)** (1,0364), seguidos pela **Temperature (cel)** (0,4177). Esse resultado sugere que a identificação das situações classificadas como adequadas está fortemente relacionada à combinação entre parâmetros físico-químicos e características do ambiente monitorado. O destaque de *Effluent* nessa classe indica que a contribuição dessa variável não está necessariamente associada a uma única direção de classificação, mas sim à sua capacidade de auxiliar o modelo na definição das fronteiras entre as classes reconstruídas.

A classe **Não Adequada** apresentou um comportamento distinto. Nessa categoria, os atributos mais relevantes foram **Nitrogen (mg/l)** (1,3406), **Orthophosphate (mg/l)** (1,1956), **Temperature (cel)** (0,2400), **Waterbody Type_River** (0,2007) e **Waterbody Type_Sewage** (0,1694). Observa-se que os nutrientes continuam exercendo papel central nas decisões do modelo, reforçando sua importância para a identificação dos padrões associados às situações de maior inadequação. Além disso, a maior contribuição de categorias como *River* e *Sewage* sugere que determinadas características ambientais e contextuais também foram utilizadas pelo algoritmo para diferenciar os casos pertencentes a essa classe.

Os resultados observados para as classes **Adequada** e **Não Adequada** são coerentes com a estratégia adotada para a reconstrução dos rótulos. Como essas categorias representam os grupos de maior confiança derivados das probabilidades do modelo binário, espera-se que sejam caracterizadas por padrões mais bem definidos e, consequentemente, por maiores valores de importância atribuídos às variáveis mais discriminativas.

A classe **Atenção**, por sua vez, apresentou um comportamento consideravelmente diferente. Os maiores valores de importância foram observados para **Orthophosphate (mg/l)** (0,5209), **Nitrogen (mg/l)** (0,3277), **Waterbody Type_Effluent** (0,2835) e **Temperature (cel)** (0,2150), porém com magnitudes significativamente inferiores às verificadas nas classes extremas. Esse resultado é particularmente relevante porque a classe Atenção foi construída justamente para representar a região intermediária de transição entre condições mais claramente adequadas e mais claramente não adequadas.

A menor concentração da importância em atributos específicos sugere que os registros pertencentes à classe Atenção apresentam características menos definidas, exigindo que o modelo utilize um conjunto mais diversificado de informações para realizar sua identificação. Esse comportamento é compatível com o próprio processo de construção da classe, formada a partir das observações localizadas próximas às fronteiras probabilísticas aprendidas pelo modelo binário.

Outro aspecto relevante é que **Orthophosphate (mg/l)** e **Nitrogen (mg/l)** permaneceram entre as variáveis mais importantes em todas as classes analisadas. Esse resultado reforça os achados obtidos na análise global e nas etapas exploratórias anteriores, indicando que esses parâmetros carregam informações fundamentais para a separação dos padrões presentes na base de dados. Entretanto, observa-se que sua influência é significativamente mais intensa nas classes extremas do que na classe intermediária, evidenciando que esses atributos foram especialmente importantes para distinguir situações classificadas com maior grau de confiança pelo modelo.

Entre as variáveis categóricas, destacam-se principalmente **Waterbody Type_Effluent**, **Waterbody Type_River** e **Waterbody Type_Sewage**, que apresentaram contribuições relevantes em diferentes classes. Contudo, essa importância deve ser interpretada com cautela, uma vez que pode refletir tanto características ambientais específicas desses corpos hídricos quanto aspectos relacionados à distribuição das amostras presentes no conjunto de treinamento. A análise SHAP permite identificar a influência dessas variáveis nas previsões, mas não possibilita determinar os mecanismos causais responsáveis por essa influência.

De maneira geral, os resultados mostram que as classes **Adequada** e **Não Adequada** são fortemente determinadas por variáveis relacionadas à concentração de nutrientes e às características dos corpos hídricos, enquanto a classe **Atenção** apresenta um padrão mais difuso e distribuído entre diferentes atributos. Esse comportamento é consistente com a lógica da reconstrução probabilística dos rótulos, na qual a classe intermediária representa justamente uma região de transição entre os extremos, caracterizada por maior ambiguidade e menor separação entre os padrões aprendidos pelo modelo.


In [24]:
fig = px.imshow(
    heatmap_data,
    text_auto=".3f",
    color_continuous_scale="Viridis",
    aspect="auto",
    title="Importância Média das Variáveis por Classe (SHAP)"
)

fig.update_layout(
    title={
        "text": "Importância Média das Variáveis por Classe (SHAP)",
        "x": 0.5
    },
    height=800
)

fig.show()

## Interpretação da Importância das Variáveis por Classe

O heatmap de importância por classe permitiu analisar como cada variável contribui para a identificação das categorias **Adequada**, **Atenção** e **Não Adequada** definidas pela estratégia de reconstrução probabilística dos rótulos. Diferentemente da análise global, essa visualização evidencia que uma mesma variável pode exercer níveis distintos de influência dependendo da classe considerada, revelando como o modelo utiliza diferentes atributos para caracterizar cada região do espaço de classificação.

O primeiro aspecto que chama atenção é a diferença entre as classes extremas (**Adequada** e **Não Adequada**) e a classe intermediária (**Atenção**). Observa-se que as maiores importâncias estão concentradas nas classes extremas, enquanto a classe Atenção apresenta valores significativamente menores para praticamente todas as variáveis. Esse comportamento é coerente com o próprio processo de construção dos rótulos, uma vez que a classe Atenção foi formada a partir das observações localizadas próximas às fronteiras probabilísticas do modelo binário, representando situações de transição entre os extremos.

Na classe **Adequada**, os maiores valores de importância foram observados para **Waterbody Type_Effluent (1,425)**, **Orthophosphate (mg/l) (1,328)** e **Nitrogen (mg/l) (1,036)**. Em seguida aparecem **Temperature (cel) (0,418)** e algumas variáveis categóricas com contribuição mais moderada. Esse resultado indica que o modelo utiliza principalmente a combinação entre características do corpo hídrico e parâmetros físico-químicos para identificar observações classificadas com maior confiança como adequadas. O destaque de *Waterbody Type_Effluent* demonstra que essa categoria exerceu forte influência sobre as decisões do modelo, embora a análise SHAP não permita determinar a direção dessa influência nem estabelecer relações causais.

A classe **Não Adequada** apresentou um padrão semelhante em termos de concentração da importância em poucas variáveis, porém com uma hierarquia diferente. Os maiores valores foram observados para **Nitrogen (mg/l) (1,341)** e **Orthophosphate (mg/l) (1,196)**, seguidos por **Temperature (cel) (0,240)**, **Waterbody Type_River (0,201)** e **Waterbody Type_Sewage (0,169)**. Esse resultado sugere que o modelo depende fortemente das variáveis relacionadas à concentração de nutrientes para identificar situações classificadas como não adequadas, utilizando as características do corpo hídrico como informações complementares para refinar essa separação.

A classe **Atenção** apresentou comportamento distinto das demais. Embora **Orthophosphate (mg/l) (0,521)**, **Nitrogen (mg/l) (0,328)**, **Waterbody Type_Effluent (0,284)** e **Temperature (cel) (0,215)** continuem sendo as variáveis mais importantes, suas contribuições são consideravelmente menores quando comparadas às observadas nas classes extremas. Esse padrão sugere que a identificação das observações pertencentes à classe Atenção depende de uma combinação mais equilibrada de atributos, sem a presença de variáveis com elevado poder discriminativo.

Esse resultado é particularmente relevante porque fornece evidências de que a classe Atenção realmente representa uma região intermediária de transição entre os extremos da classificação. Como essa categoria foi construída a partir das observações localizadas próximas às fronteiras probabilísticas aprendidas pelo modelo binário, espera-se que seus padrões sejam menos definidos e mais difíceis de separar. A distribuição mais homogênea das importâncias observadas no heatmap reforça essa interpretação.

Outro aspecto relevante é o comportamento consistente de **Orthophosphate (mg/l)** e **Nitrogen (mg/l)** em todas as classes analisadas. Essas variáveis permanecem entre as mais importantes independentemente da categoria considerada, reforçando os resultados obtidos na análise global e nas etapas exploratórias anteriores. Entretanto, observa-se que sua influência é significativamente maior nas classes extremas do que na classe Atenção, indicando que esses atributos desempenham papel fundamental na definição das fronteiras que separam observações claramente adequadas das claramente não adequadas.

Entre as variáveis categóricas, destacam-se principalmente **Waterbody Type_Effluent**, **Waterbody Type_River** e **Waterbody Type_Sewage**, que apresentaram contribuições relevantes em diferentes classes. Contudo, essa importância deve ser interpretada com cautela, uma vez que pode refletir tanto características ambientais específicas desses corpos hídricos quanto aspectos relacionados à distribuição das amostras presentes na base de dados.

De maneira geral, os resultados indicam que o LightGBM aprendeu padrões distintos para cada classe da estrutura reconstruída. Enquanto as categorias Adequada e Não Adequada são fortemente caracterizadas por um conjunto reduzido de variáveis altamente discriminantes, a classe Atenção apresenta um padrão mais difuso e distribuído entre diferentes atributos. Esse comportamento é consistente com a lógica utilizada na reconstrução dos rótulos e fornece evidências adicionais de que a nova estrutura de classes capturou diferentes níveis de confiança presentes nas previsões do modelo.


In [25]:
top_features_por_classe = []

for classe in heatmap_data.columns:
    top_classe = (
        heatmap_data[[classe]]
        .sort_values(by=classe, ascending=False)
        .head(5)
        .reset_index()
    )

    top_classe.columns = ["Variável", "Importância SHAP"]
    top_classe["Classe"] = classe

    top_features_por_classe.append(top_classe)

top_features_por_classe = pd.concat(
    top_features_por_classe,
    ignore_index=True
)

top_features_por_classe

fig = px.bar(
    top_features_por_classe,
    x="Importância SHAP",
    y="Variável",
    color="Classe",
    facet_col="Classe",
    orientation="h",
    text="Importância SHAP",
    title="Top 5 Variáveis Mais Importantes por Classe",
    color_discrete_sequence=[
        "#004D48",
        "#66CDAA",
        "#7FFFD4"
    ]
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    marker_line_color="black",
    marker_line_width=0.4
)

fig.update_layout(
    title={
        "text": "Top 5 Variáveis Mais Importantes por Classe",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20}
    },
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#222222"),
    height=650,
    width=1300,
    showlegend=False
)

fig.update_yaxes(
    categoryorder="total ascending"
)

fig.show()

## Interpretação do Top 5 por Classe

O gráfico das cinco variáveis mais importantes por classe reforça os padrões observados anteriormente na análise do heatmap. Observa-se que as classes **Adequada** e **Não Adequada** apresentam variáveis com elevada capacidade discriminativa, enquanto a classe **Atenção** possui importâncias mais distribuídas e de menor magnitude.

Na classe **Adequada**, destacam-se principalmente **Waterbody Type_Effluent**, **Orthophosphate (mg/l)** e **Nitrogen (mg/l)**, seguidos por **Temperature (cel)** e **Waterbody Type_Sewage**. A elevada importância dessas variáveis sugere que o modelo utiliza uma combinação entre características físico-químicas e informações contextuais dos corpos hídricos para identificar observações classificadas com maior confiança como adequadas.

Na classe **Não Adequada**, os maiores destaques são **Nitrogen (mg/l)** e **Orthophosphate (mg/l)**, seguidos por **Temperature (cel)**, **Waterbody Type_River** e **Waterbody Type_Sewage**. A predominância dos nutrientes nessa classe indica que essas variáveis desempenham papel central na identificação dos padrões associados às situações classificadas como não adequadas, evidenciando sua relevância para a separação das observações localizadas nos extremos da estrutura reconstruída.

Por outro lado, a classe **Atenção** apresenta um comportamento distinto. Embora **Orthophosphate (mg/l)**, **Nitrogen (mg/l)**, **Waterbody Type_Effluent**, **Temperature (cel)** e **Country_USA** apareçam entre as variáveis mais importantes, seus valores de importância são consideravelmente inferiores aos observados nas classes extremas. Esse resultado sugere que a identificação dessa categoria depende de um conjunto mais amplo de informações, sem a presença de atributos com elevado poder discriminativo.

Esse comportamento é coerente com a própria construção da classe Atenção. Como essa categoria foi formada a partir das observações localizadas próximas às fronteiras probabilísticas do modelo binário, espera-se que seus padrões sejam menos definidos e apresentem maior sobreposição com as demais classes. Consequentemente, o modelo precisa combinar diferentes atributos para realizar sua identificação, em vez de depender fortemente de poucas variáveis dominantes.

De maneira geral, o gráfico evidencia que as classes **Adequada** e **Não Adequada** são caracterizadas por padrões mais bem definidos e sustentados por variáveis altamente discriminativas, enquanto a classe **Atenção** representa uma região intermediária de transição, marcada por maior ambiguidade e por uma distribuição mais equilibrada da importância entre os atributos utilizados pelo modelo.


# Conclusão da Análise de Interpretabilidade do Modelo

A etapa de interpretabilidade representa o encerramento do processo de investigação desenvolvido ao longo desta pesquisa. Os experimentos iniciais evidenciaram que os algoritmos apresentavam dificuldades para separar adequadamente determinadas categorias de qualidade da água, especialmente aquelas associadas a condições intermediárias. Embora alguns modelos alcançassem resultados satisfatórios em métricas globais, análises mais detalhadas mostraram que a estrutura original dos rótulos continha regiões de forte sobreposição, dificultando a construção de fronteiras de decisão bem definidas.

A partir dessas observações, foram conduzidas análises exploratórias complementares que revelaram limitações importantes na distribuição das classes e na separação dos padrões presentes nos dados. Esse cenário motivou a reformulação da estratégia de classificação adotada no trabalho. Em vez de utilizar exclusivamente os rótulos originalmente definidos, foi empregada uma abordagem baseada em classificação binária seguida de reconstrução probabilística das classes. Nesse processo, as probabilidades produzidas pelo modelo foram utilizadas para identificar regiões de maior confiança classificatória e regiões de transição, resultando na estrutura final composta pelas classes **Adequada**, **Atenção** e **Não Adequada**.

Os resultados obtidos durante a etapa de modelagem demonstraram que essa nova representação permitiu uma separação mais coerente dos padrões presentes na base de dados. Entre os algoritmos avaliados, o LightGBM apresentou o melhor equilíbrio entre desempenho preditivo, capacidade de generalização e consistência na identificação das classes reconstruídas, sendo selecionado como modelo final do sistema.

Entretanto, a seleção de um modelo não deve se basear exclusivamente em métricas de desempenho. Mesmo quando um algoritmo apresenta resultados satisfatórios de acurácia, precisão, recall e F1-score, permanece a necessidade de compreender quais fatores influenciam suas decisões. Nesse contexto, a aplicação do SHAP permitiu adicionar uma camada de transparência ao processo de classificação, tornando possível investigar quais variáveis exerceram maior influência sobre as previsões realizadas pelo LightGBM.

A análise de importância global mostrou que **Orthophosphate (mg/l)** e **Nitrogen (mg/l)** foram os atributos mais relevantes para o comportamento geral do modelo, seguidos por **Waterbody Type_Effluent** e **Temperature (cel)**. Esse resultado mostrou-se consistente com os achados obtidos durante a análise exploratória dos dados, na qual essas variáveis já haviam se destacado entre os parâmetros mais associados às variações observadas no índice CCME. Além disso, a literatura especializada reconhece a relevância desses nutrientes em processos relacionados à qualidade da água, reforçando a plausibilidade dos padrões aprendidos pelo algoritmo.

A análise por classe permitiu aprofundar ainda mais essa compreensão. Observou-se que as classes **Adequada** e **Não Adequada** apresentam forte dependência de um conjunto reduzido de variáveis altamente discriminativas, especialmente Orthophosphate, Nitrogen, Temperature e determinadas categorias de corpos hídricos. Em contrapartida, a classe **Atenção** apresentou importâncias significativamente menores e mais distribuídas entre os atributos analisados.

Esse comportamento é particularmente relevante porque está diretamente relacionado à forma como os rótulos foram reconstruídos. Como a classe Atenção foi formada a partir das observações localizadas próximas às fronteiras probabilísticas aprendidas pelo modelo binário, espera-se que seus padrões sejam menos definidos e apresentem maior sobreposição com as demais categorias. Os resultados do SHAP confirmam essa hipótese, indicando que a identificação dessa classe depende de uma combinação mais ampla de características, enquanto as classes extremas são sustentadas por atributos com maior capacidade discriminativa.

Outro aspecto importante desta análise é a convergência observada entre diferentes etapas da pesquisa. Os padrões identificados pelo SHAP são compatíveis com os resultados das análises exploratórias, com o comportamento observado durante os experimentos de classificação e com o conhecimento já consolidado na literatura especializada. Embora essa convergência não permita estabelecer relações causais, ela fornece evidências de que o modelo aprendeu padrões coerentes com a estrutura dos dados e com fatores reconhecidamente relevantes para a avaliação da qualidade da água.

Além disso, a análise de interpretabilidade fornece evidências adicionais de que a estratégia de reconstrução dos rótulos foi capaz de capturar diferentes níveis de confiança presentes nas previsões do modelo. A distinção observada entre as classes extremas e a classe intermediária sugere que a nova estrutura de classificação não apenas melhorou o desempenho preditivo, mas também produziu categorias mais compatíveis com a organização dos padrões efetivamente presentes nos dados.

Por fim, os resultados obtidos reforçam a adequação da escolha do LightGBM como modelo final para o problema estudado. Além de apresentar elevado desempenho e boa capacidade de generalização, o algoritmo demonstrou basear suas decisões em variáveis consistentes com os achados das etapas anteriores da pesquisa. Dessa forma, a análise de interpretabilidade não apenas aumenta a transparência das previsões realizadas pelo modelo, mas também fortalece a confiança nos resultados obtidos, contribuindo para que futuras aplicações do sistema possam ser realizadas com maior robustez, rastreabilidade e embasamento científico.


In [26]:
pareto_shap = shap_importance.copy()

pareto_shap = pareto_shap.sort_values(
    by="Importância SHAP",
    ascending=False
).reset_index(drop=True)

pareto_shap["Percentual"] = (
    pareto_shap["Importância SHAP"] / pareto_shap["Importância SHAP"].sum()
) * 100

pareto_shap["Percentual Acumulado"] = pareto_shap["Percentual"].cumsum()

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=pareto_shap["Variável"],
        y=pareto_shap["Percentual"],
        name="Importância individual (%)",
        text=pareto_shap["Percentual"],
        texttemplate="%{text:.1f}%",
        textposition="outside",
        marker=dict(
            color="#004D48",
            line=dict(color="black", width=0.5)
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=pareto_shap["Variável"],
        y=pareto_shap["Percentual Acumulado"],
        name="Importância acumulada (%)",
        mode="lines+markers+text",
        text=pareto_shap["Percentual Acumulado"],
        texttemplate="%{text:.1f}%",
        textposition="top center",
        yaxis="y2",
        line=dict(
            color="#F4D03F",
            width=3
        ),
        marker=dict(
            size=8,
            color="#F4D03F",
            line=dict(color="black", width=0.5)
        )
    )
)

fig.add_hline(
    y=80,
    line_dash="dash",
    line_color="#E74C3C",
    annotation_text="80%",
    annotation_position="top left",
    secondary_y=False
)

fig.update_layout(
    title={
        "text": "Diagrama de Pareto da Importância Global das Variáveis (SHAP)",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20}
    },
    xaxis=dict(
        title="Variáveis",
        tickangle=-45
    ),
    yaxis=dict(
        title="Importância individual (%)",
        range=[0, max(pareto_shap["Percentual"]) + 10],
        showgrid=True,
        gridcolor="#E5E5E5"
    ),
    yaxis2=dict(
        title="Importância acumulada (%)",
        overlaying="y",
        side="right",
        range=[0, 105],
        showgrid=False
    ),
    legend=dict(
        orientation="h",
        y=1.12,
        x=0.5,
        xanchor="center"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="#222222"),
    height=650,
    width=1200,
    margin=dict(l=70, r=80, t=120, b=160)
)

fig.show()

## Interpretação do Diagrama de Pareto

O Diagrama de Pareto foi utilizado para analisar como a importância das variáveis está distribuída ao longo do modelo e identificar quais atributos concentram a maior parcela da capacidade preditiva do LightGBM. Os resultados mostram que a contribuição das variáveis não ocorre de forma uniforme, estando fortemente concentrada em um pequeno conjunto de características.

Observa-se que **Orthophosphate (mg/l)**, **Nitrogen (mg/l)** e **Waterbody Type_Effluent** são responsáveis, em conjunto, por aproximadamente **77,8%** da importância total calculada pelo SHAP. Quando são consideradas também as variáveis **Temperature (cel)** e **Waterbody Type_River**, a importância acumulada alcança aproximadamente **90,3%**. Isso significa que apenas cinco das dezenove variáveis utilizadas pelo modelo concentram praticamente toda a informação empregada pelo algoritmo para realizar suas previsões.

Entre essas variáveis, destaca-se principalmente **Orthophosphate (mg/l)**, responsável por cerca de **31,3%** da importância total, seguida por **Nitrogen (mg/l)** com **27,8%** e **Waterbody Type_Effluent** com **18,7%**. Esse resultado reforça os achados obtidos nas análises anteriores, nas quais os nutrientes já haviam se destacado como os atributos mais relevantes para a separação das classes reconstruídas. Da mesma forma, a elevada contribuição da categoria *Effluent* indica que determinadas características associadas ao tipo de corpo hídrico também desempenham papel importante na estrutura decisória aprendida pelo modelo.

Outro aspecto relevante é a rápida estabilização da curva de importância acumulada. Após a inclusão das cinco variáveis mais importantes, os ganhos proporcionados pelos demais atributos tornam-se progressivamente menores. Isso sugere que o modelo concentra suas decisões em um conjunto reduzido de características altamente informativas, enquanto as demais variáveis exercem papel complementar no refinamento das fronteiras de classificação.

Esse comportamento é consistente com os resultados obtidos nas análises globais e por classe do SHAP. As variáveis que aparecem nas primeiras posições do Diagrama de Pareto são as mesmas que apresentaram maior influência sobre as classes **Adequada**, **Atenção** e **Não Adequada**, evidenciando a consistência dos padrões identificados ao longo das diferentes etapas de interpretabilidade.

De maneira geral, o Diagrama de Pareto reforça que o LightGBM aprendeu uma estrutura decisória relativamente concentrada, baseada principalmente em informações relacionadas à concentração de nutrientes, às características dos corpos hídricos e às condições ambientais observadas. Essa concentração da importância em um conjunto reduzido de atributos contribui para aumentar a interpretabilidade do modelo e fornece evidências adicionais de que as previsões estão fundamentadas em variáveis que demonstraram relevância ao longo de toda a investigação realizada neste trabalho.


In [30]:
# =====================================
# DADOS SHAP — TOP 5 VARIÁVEIS POR CLASSE
# =====================================

classes_data = {
    "Adequada": {
        "pred": 0.87,
        "base": 0.34,
        "features": [
            ("Waterbody Type_Effluent", 1.425),
            ("Orthophosphate (mg/l)",  -1.328),
            ("Nitrogen (mg/l)",        -1.036),
            ("Temperature (cel)",       0.418),
            ("Waterbody Type_Sewage",   0.079),
        ]
    },
    "Atenção": {
        "pred": 0.61,
        "base": 0.50,
        "features": [
            ("Orthophosphate (mg/l)",  -0.521),
            ("Nitrogen (mg/l)",        -0.328),
            ("Waterbody Type_Effluent", 0.284),
            ("Temperature (cel)",       0.215),
            ("Country_USA",             0.133),
        ]
    },
    "Não Adequada": {
        "pred": 0.72,
        "base": 0.16,
        "features": [
            ("Nitrogen (mg/l)",         1.341),
            ("Orthophosphate (mg/l)",   1.196),
            ("Temperature (cel)",      -0.240),
            ("Waterbody Type_River",    0.201),
            ("Waterbody Type_Sewage",   0.169),
        ]
    }
}

# =====================================
# CORES AQUASENSE
# =====================================

COR_POS   = "#3FF3D7"
COR_NEG   = "#C85E3A"
COR_TEXTO = "#004D48"

# =====================================
# SUBPLOTS — 1 LINHA POR CLASSE
# =====================================

n_classes = len(classes_data)
fig = make_subplots(
    rows=n_classes,
    cols=1,
    subplot_titles=list(classes_data.keys()),
    vertical_spacing=0.14,
)

X_MIN = -1.8
X_MAX = 2.2
BAR_WIDTH = 0.5

for row, (classe, dados) in enumerate(classes_data.items(), start=1):

    features = dados["features"]
    base_val = dados["base"]
    pred_val = dados["pred"]

    # Ordena do menor para o maior valor SHAP (barras negativas embaixo)
    sorted_features = sorted(features, key=lambda x: x[1])

    # Acumula a partir do valor base para montar o waterfall
    x_start = base_val

    for feat_name, shap_val in sorted_features:
        cor = COR_POS if shap_val >= 0 else COR_NEG
        sinal = "+" if shap_val >= 0 else ""

        # Posiciona o rótulo para não colidir com a barra
        if shap_val >= 0:
            text_pos = "outside"
        else:
            text_pos = "outside"

        fig.add_trace(
            go.Bar(
                x=[shap_val],
                y=[feat_name],
                orientation="h",
                marker_color=cor,
                marker_line=dict(color="white", width=0.5),
                name=f"{sinal}{shap_val:.3f}",
                text=f"{sinal}{shap_val:.3f}",
                textposition="outside",
                textfont=dict(size=10, color=COR_TEXTO),
                base=x_start,
                showlegend=False,
                width=BAR_WIDTH,
            ),
            row=row,
            col=1,
        )
        x_start += shap_val

    # Linha vertical do valor base
    fig.add_vline(
        x=base_val,
        line_dash="dot",
        line_color="#888888",
        line_width=1.2,
        row=row,
        col=1,
    )

    # --- Anotação do valor base: acima do plot, alinhada à linha ---
    # Usamos yref do domínio do subplot para posicionar fora das barras
    y_domain_map = {1: [0.72, 1.0], 2: [0.36, 0.64], 3: [0.0, 0.28]}
    y_top = y_domain_map[row][1]
    y_bot = y_domain_map[row][0]

    # base label — dentro do painel, na primeira barra (y=0 no eixo de dados)
    fig.add_annotation(
        x=base_val,
        xref=f"x{row if row > 1 else ''}",
        y=len(features) - 1,
        yref=f"y{row if row > 1 else ''}",
        text=f"base = {base_val:.2f}",
        showarrow=False,
        font=dict(size=10, color="#888888"),
        xanchor="left",
        yanchor="bottom",
        bgcolor="white",
        borderpad=2,
    )

    # f(x) label — abaixo do painel
    fig.add_annotation(
        x=pred_val,
        xref=f"x{row if row > 1 else ''}",
        y=y_bot + 0.005,
        yref="paper",
        text=f"<b>f(x) = {pred_val:.2f}</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor=COR_TEXTO,
        ax=0,
        ay=20,
        font=dict(size=11, color=COR_TEXTO),
        xanchor="center",
        yanchor="bottom",
        bgcolor="white",
        borderpad=2,
    )

# =====================================
# LAYOUT
# =====================================

fig.update_layout(
    title={
        "text": "Force Plot SHAP — Top 5 Variáveis por Classe",
        "x": 0.5,
        "y": 0.98,
        "font": dict(size=20, color=COR_TEXTO),
    },
    height=820,
    width=1080,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=12, color=COR_TEXTO),
    margin=dict(l=220, r=130, t=100, b=60),
    bargap=0.3,
)

fig.update_xaxes(
    title_text="Valor SHAP",
    showgrid=True,
    gridcolor="#DDEFEA",
    zeroline=True,
    zerolinecolor="#AAAAAA",
    zerolinewidth=1.5,
    range=[X_MIN, X_MAX],
    tickfont=dict(size=10),
)

fig.update_yaxes(
    tickfont=dict(size=11),
)

# Legenda manual (anotação no topo)
fig.add_annotation(
    text=(
        "<span style='color:#3FF3D7'>■</span> Empurra para esta classe"
        "   <span style='color:#C85E3A'>■</span> Empurra para longe desta classe"
    ),
    xref="paper", yref="paper",
    x=0.5, y=1.045,
    showarrow=False,
    font=dict(size=12),
    xanchor="center",
    yanchor="bottom",
)

fig.show()

# Contribuição das Variáveis na Classificação de Qualidade Hídrica
O force plot SHAP revela como cada variável empurra a predição do modelo para mais perto ou mais longe de cada classe, partindo de um valor base comum.

Na classe **Adequada**, o tipo de corpo hídrico *Effluent* é o fator dominante, com impacto de +1,43 — o maior valor absoluto em todo o gráfico. Ele sozinho é responsável por levar a predição muito acima do valor base de 0,34, chegando a 0,87. As concentrações de ortofosfato e nitrogênio atuam na direção contrária, com −1,33 e −1,04 respectivamente, mas não são suficientes para reverter a classificação. O resultado é uma predição alta, mas com tensão interna considerável entre as forças opostas.

Na classe **Atenção**, os impactos são os mais equilibrados e de menor magnitude entre as três classes. Nenhuma variável exerce força dominante: o maior impacto positivo é +0,28 (*Effluent*) e o maior negativo é −0,52 (ortofosfato). Esse equilíbrio de forças explica por que o modelo sai do valor base de 0,50 e avança apenas para 0,61 — a menor variação relativa ao base entre as três classes, indicando que o modelo distribui a incerteza justamente nessa categoria intermediária.

Na classe **Não Adequada**, o padrão se inverte completamente em relação à classe Adequada. Nitrogênio e ortofosfato, que antes empurravam para baixo, tornam-se as forças mais intensas na direção positiva: +1,34 e +1,20. O valor base já é o mais baixo dos três (0,16), mas a força combinada dessas variáveis eleva a predição a 0,72. Os tipos *River* e *Sewage* somam impulsos adicionais menores, enquanto a temperatura é a única variável que puxa na direção contrária, com −0,24 — o único freio nessa classe.

Lendo os três painéis em conjunto, o que o force plot mostra com clareza é que ortofosfato e nitrogênio funcionam como uma alavanca bidirecional: quando seus valores são altos, afastam o modelo da conformidade e o aproximam da não adequação. O tipo *Effluent* opera de forma complementar, favorecendo as classes positivas. A temperatura aparece nos três painéis mas sempre com papel secundário, sem força suficiente para inverter a direção determinada pelos nutrientes.

In [27]:
# ============================================
# BEESWARM SHAP — CLASSE NÃO ADEQUADA (idx=2)
# ============================================

CLASS_IDX = 2  # 0=Adequada, 1=Atenção, 2=Não adequada
CLASS_NAME = "Não Adequada"
N_FEATURES = 10  # top N variáveis

feature_names_clean = [
    f.replace("remainder__", "").replace("cat__", "").replace("Waterbody Type_", "WB: ").replace("Country_", "País: ")
    for f in preprocessor.get_feature_names_out()
]

shap_vals = shap_values.values[:, :, CLASS_IDX]  # (n_samples, n_features)
X_vals    = X_test_transformed_final             # (n_samples, n_features)

# Ordenar por importância média absoluta
mean_abs = np.abs(shap_vals).mean(axis=0)
top_idx  = np.argsort(mean_abs)[::-1][:N_FEATURES]
top_idx  = top_idx[::-1]  # menor importância no topo → maior embaixo (visual padrão)

# Beeswarm: distribuir pontos em y com jitter para evitar sobreposição
np.random.seed(42)

fig = go.Figure()

for rank, feat_i in enumerate(top_idx):
    y_base   = rank
    shap_col = shap_vals[:, feat_i]
    feat_col = X_vals[:, feat_i]

    # Normalizar cor da feature [0,1]
    feat_min, feat_max = feat_col.min(), feat_col.max()
    if feat_max > feat_min:
        feat_norm = (feat_col - feat_min) / (feat_max - feat_min)
    else:
        feat_norm = np.zeros_like(feat_col)

    # Jitter vertical
    jitter = np.random.uniform(-0.3, 0.3, size=len(shap_col))
    y_vals = y_base + jitter

    # Cor: azul escuro (baixo) → verde água (alto)
    colors = [
        f"rgb({int(0 + (63 - 0) * v)}, {int(77 + (243 - 77) * v)}, {int(72 + (215 - 72) * v)})"
        for v in feat_norm
    ]

    fig.add_trace(go.Scatter(
        x=shap_col,
        y=y_vals,
        mode="markers",
        marker=dict(
            size=4,
            color=feat_norm,
            colorscale=[
                [0.0, "#004D48"],
                [0.5, "#1a9b8f"],
                [1.0, "#3FF3D7"]
            ],
            opacity=0.6,
            showscale=(rank == len(top_idx) - 1),  # colorbar só 1x
            colorbar=dict(
                title="Valor da<br>feature",
                tickvals=[0, 1],
                ticktext=["Baixo", "Alto"],
                thickness=14,
                len=0.5,
                x=1.02
            )
        ),
        name=feature_names_clean[feat_i],
        showlegend=False,
        hovertemplate=(
            f"<b>{feature_names_clean[feat_i]}</b><br>"
            "SHAP: %{x:.4f}<br>"
            "Feature value: %{customdata:.4f}<extra></extra>"
        ),
        customdata=feat_col
    ))

fig.add_vline(x=0, line_dash="dash", line_color="#888888", line_width=1)

fig.update_layout(
    title={
        "text": f"SHAP Beeswarm Plot — Classe: {CLASS_NAME}",
        "x": 0.5,
        "font": dict(size=22, color="#004D48")
    },
    xaxis=dict(
        title="Valor SHAP (impacto na previsão)",
        showgrid=True,
        gridcolor="#DDEFEA",
        zeroline=False
    ),
    yaxis=dict(
        tickvals=list(range(N_FEATURES)),
        ticktext=[feature_names_clean[i] for i in top_idx],
        showgrid=False
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13, color="#004D48"),
    height=600,
    width=1100,
    margin=dict(l=200, r=100, t=80, b=60)
)

fig.show()

# Beeswarm Plot — Classe: Não Adequada

O beeswarm plot complementa o force plot ao mostrar não apenas a direção e magnitude do impacto de cada variável, mas também como esse impacto se distribui ao longo de todo o conjunto de amostras — revelando padrões que um único ponto de predição não captura.

O nitrogênio é, de longe, a variável mais influente. Sua nuvem de pontos se estende da esquerda até valores acima de 6, com a grande maioria dos pontos concentrada no lado positivo e em tonalidade clara (valores altos da feature). Isso significa que, quando a concentração de nitrogênio é elevada, o modelo aumenta substancialmente a probabilidade de classificar a amostra como não adequada, e esse comportamento é consistente em praticamente todas as amostras. A dispersão ampla indica que o efeito não é uniforme: concentrações muito altas geram impactos SHAP extremamente elevados, enquanto concentrações baixas (pontos escuros) se agrupam próximos ao zero ou levemente negativos.

O ortofosfato apresenta comportamento análogo ao nitrogênio, mas com distribuição ligeiramente mais compacta. Os pontos claros (valores altos) se concentram no lado positivo, e os pontos escuros (valores baixos) no negativo, confirmando uma relação monotônica clara: quanto maior a concentração, maior o impacto na direção da não adequação. A presença de pontos escuros à esquerda indica que amostras com baixo ortofosfato chegam a reduzir a probabilidade dessa classe.

A temperatura exibe um padrão invertido em relação aos nutrientes. Os pontos claros — temperaturas altas — concentram-se no lado negativo do eixo SHAP, enquanto temperaturas baixas (pontos escuros) aparecem levemente no lado positivo. Isso sugere que água mais quente, dentro desse conjunto de dados, está associada a menor probabilidade de não adequação — possivelmente por refletir condições ambientais distintas das fontes de contaminação mais críticas.

Entre os tipos de corpo hídrico, o *Sewage* chama atenção pela distribuição assimétrica: a maioria dos pontos se agrupa próxima ao zero, mas há uma cauda longa de pontos claros atingindo valores SHAP acima de 4. Isso indica que, quando o corpo hídrico é identificado como esgoto, o impacto pode ser extremamente elevado em casos específicos — não é uma influência constante, mas pontualmente decisiva. O *River* concentra seus pontos entre −0,5 e +1, com distribuição mais homogênea, atuando como fator moderado e consistente a favor da não adequação. *Effluent* e *Estuarine* apresentam pontos claros no lado negativo, indicando que esses tipos de corpo hídrico tendem a reduzir a probabilidade dessa classe quando presentes.

As variáveis de menor impacto geral — *Drainage*, *Sea Water* e *País: Ireland* — têm suas nuvens concentradas próximas ao zero, com dispersão mínima, sinalizando contribuição marginal na discriminação dessa classe.

Lendo o beeswarm em conjunto com o force plot, a conclusão se reforça: nitrogênio e ortofosfato não apenas têm os maiores impactos médios, como também apresentam a maior variabilidade de efeito entre amostras. São eles que determinam o grau de confiança do modelo na classificação de não adequação — e o fazem de forma coerente com a química aquática subjacente.